# Détection des SWR par machine learning

CA1 bipolaire + EMG à 1 kHz, comme la détection CWT, mais la détection passe par le modèle CNN+BiLSTM du labo au lieu du seuil sur la CWT. Le calcul tourne sur ce PC ou sur le cluster (paramètre `host`, pour les longues sessions) ; l'affichage et la comparaison restent sur ce PC.

Le CSV CWT existant n'est jamais modifié.

## 0. Installation (une seule fois, puis à chaque mise à jour)

Prérequis : accès SSH au GitLab du labo déjà configuré.

1. Choisir comme kernel l'environnement `.conda` (en haut à droite).
2. Exécuter la cellule ci-dessous. La première fois, elle télécharge torch, ce qui peut prendre quelques minutes.
3. **Redémarrer le kernel** (bouton *Restart*), puis passer à la section 1.

Pour une mise à jour, remplacer la version (`v0.16.1`) dans la cellule ci-dessous, dans `requirements.txt` **et** dans la commande du point D de la section 0b, puis refaire 2 et 3 et le point D : ce PC et le cluster doivent avoir la même version.

In [ ]:
import os
import sys
assert sys.version_info >= (3, 11), f"Python {sys.version.split()[0]} : il faut Python 3.11 ou plus, prévenir Mélodie"
%pip freeze > env_avant_swr.txt
# accepte la clé du serveur GitLab à la première connexion (%pip ne peut pas répondre "yes"), refuse toujours une clé qui changerait
os.environ["GIT_SSH_COMMAND"] = "ssh -o StrictHostKeyChecking=accept-new"
%pip install "haylab-tools[swr,cluster] @ git+ssh://git@10.69.168.17/haygroup/haylab-tools.git@v0.16.1"

## 0b. Préparer le cluster (une seule fois)

À faire dans un **terminal PowerShell** (*Terminal > New Terminal* dans VS Code), sauf le point D qui se fait sur le cluster.

**A. Signer ses commits à son nom.** Ce PC est partagé (session `AudreyHay`) : l'identité git se règle pour ce repo seulement. Dans le dossier `HayLabAnalysis` :

```powershell
git config --show-origin user.name     # d'où vient le nom actuel
git config user.name "CarlaBM20"
git config user.email "deski.burnetmerlincarla@gmail.com"
```

**B. Se connecter au cluster sans mot de passe** (le notebook doit pouvoir le faire seul). On réutilise la clé créée pour GitLab :

```powershell
type $HOME\.ssh\id_ed25519.pub | ssh carla.burnet-merlin@10.69.168.93 "mkdir -p ~/.ssh; chmod 700 ~/.ssh; cat >> ~/.ssh/authorized_keys; chmod 600 ~/.ssh/authorized_keys"
notepad $HOME\.ssh\config
```

(le mot de passe du cluster est demandé une dernière fois). Dans le fichier `config`, ajouter puis enregistrer :

```
Host CRNLcluster
    HostName 10.69.168.93
    User carla.burnet-merlin
```

Vérifier : `ssh -o BatchMode=yes CRNLcluster hostname` doit répondre sans rien demander.

**C. Accès au GitLab depuis le cluster.** Se connecter (`ssh CRNLcluster`) puis :

```bash
ssh -T git@10.69.168.17
```

Si la réponse n'est pas `Welcome to GitLab`, créer une clé pour le cluster et l'ajouter dans GitLab (avatar > *Preferences* > *SSH Keys*) :

```bash
ssh-keygen -t ed25519 -C "carla-cluster"
cat ~/.ssh/id_ed25519.pub
```

**D. Environnement sur le cluster** (pour envoyer les longues sessions au cluster ; relancer la même commande `pip install` à chaque mise à jour), toujours connectée au cluster (l'installation se fait sur un nœud de calcul, pas sur le nœud d'entrée) :

```bash
srun -p CPU -c 2 --mem=8G --pty bash
/usr/bin/python3 -m venv ~/.venvs/haylab_tools
~/.venvs/haylab_tools/bin/pip install "haylab-tools[swr,cluster] @ git+ssh://git@10.69.168.17/haygroup/haylab-tools.git@v0.16.1"
exit
```

## 1. Utilisation

In [1]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import signal
from IPython.display import display
from ipyfilechooser import FileChooser

import haylab_analysis
from haylab_analysis.ephys import collect_swr, detect_swr

print('haylab-tools', haylab_analysis.__version__)

haylab-tools 0.14.0


### Choix de la session et chargement des coordonnées LFP

Dans le sélecteur, aller **jusqu'au dossier qui contient le fichier brut** (`continuous.dat`, ou `DataFrame_rawdataDS.pkl` / `RawDataChannelExtractedDS.npy` s'ils existent), puis cliquer sur **Select**. Exemple :

`...\Record Node 101\experiment1\recording2\continuous\OE_FPGA_Acquisition_Board-100.Rhythm Data`

Le fichier `structure.oebin` est lu automatiquement deux niveaux au-dessus (dossier `recording2`).

In [2]:
default_dpath = r"\\10.69.168.1\crnldata\forgetting\Carla"

%store -r dpath
# ignore une valeur stockée invalide (ancien fichier .pkl, dossier disparu...)
if "dpath" not in globals() or not os.path.isdir(str(dpath)):
    dpath = default_dpath
if not os.path.isdir(dpath):
    print("Dossier par défaut inaccessible (VPN / lecteur réseau ?), départ depuis le dossier personnel")
    dpath = os.path.expanduser("~")

fc1 = FileChooser(dpath, select_default=True, show_only_dirs=True,
                  title="<b>Go inside the folder containing the LFP raw file</b>")
display(fc1)

def update_my_folder(chooser):
    global dpath
    if chooser.selected is None:
        return
    dpath = chooser.selected
    %store dpath

fc1.register_callback(update_my_folder)

FileChooser(path='\\10.69.168.1\crnldata\forgetting\Carla\OpenEphysData\Sleep\10 mois\CarreRouge_CarreMarron_2…

In [4]:
# CSV des coordonnées LFP de toutes les souris (une ligne par souris, une colonne par région : CA1, EMG, ...)
Channels = r"C:\Users\AudreyHay\Documents\Carla\Visual Code\HayLabAnalysis\python\_LFP_coordinates_of_all_mice.csv"
assert os.path.isfile(Channels), f"CSV introuvable : {Channels}"

all_LFPcoordinates = pd.read_csv(Channels, sep=';', index_col=0).astype(str)
print("Souris dans le CSV :", list(all_LFPcoordinates.index))
print("Régions :", list(all_LFPcoordinates.columns))

Souris dans le CSV : ['TraitRouge', 'TraitBleu', 'TraitViolet', 'TraitJaune', 'TraitVert', 'TraitOrange', 'RondRouge', 'RondOrange', 'RondJaune', 'RondViolet', 'RondVert', 'RondBleu', 'RondMarron', 'RondNoir', 'TriangleBlanc', 'TriangleViolet', 'TriangleJaune', 'TriangleOrange', 'TriangleVert', 'TriangleBleu', 'TriangleMarron', 'TriangleRouge', 'TriangleNoir', 'CarreRouge', 'CarreMarron', 'CarreNoir', 'CarreBlanc', 'CarreVert', 'CarreViolet', 'CarreJaune', 'CarreOrange', 'EtoileBleu', 'EtoileJaune', 'EtoileViolet', 'EtoileOrange', 'EtoileBlanc', 'EtoileNoir']
Régions : ['EMG', 'S1', 'PFC', 'CA1', 'RSC', 'oPFC', 'EnthC', 'RH']


### Paramètres

Commencer avec `test_minutes = 10` : la cellule *Détection* affiche une estimation du temps de calcul pour la session entière. Ensuite, mettre `test_minutes = None` et tout relancer.

`host = 'local'` calcule sur ce PC. `host = 'CRNLcluster'` envoie le calcul au cluster (GPU si disponible) : nécessaire pour les longues sessions, une fois la section 0b faite.

La souris est trouvée **automatiquement** à partir du nom du dossier (comparé aux noms du CSV). `id_in_path` et `ID_choice` ne servent que quand le dossier contient plusieurs noms de souris.

In [7]:
file_suffix = 'CarreRouge'    # suffixe ajouté aux noms de fichiers, comme dans SWRextraction_Carla
test_minutes = 10   # None = toute la session. Commencer par 10 min pour mesurer le temps de calcul
fs = 1000           # fréquence attendue par le modèle (le signal est rééchantillonné si besoin)
host = 'local'      # 'local' = sur ce PC ; 'CRNLcluster' = sur le cluster (longues sessions)
cluster_activate = 'source ~/.venvs/haylab_tools/bin/activate'   # environnement du cluster (section 0b, point D)

# Plusieurs noms de souris dans le chemin MAIS une seule souris enregistrée (canaux < 32) :
id_in_path = 1      # 0 = 1er nom trouvé dans le chemin, 1 = 2e nom, etc.
# Plusieurs souris enregistrées en même temps (canaux >= 32) :
ID_choice = 0       # entrée Open Ephys : 0 = 1re souris, 1 = 2e, 2 = 3e, 3 = 4e

suffix = f"_ml{file_suffix}" + (f"_test{test_minutes}min" if test_minutes else "")

### Chargement du fichier brut, identification de la souris, CA1 bipolaire + EMG

Le modèle reçoit le signal **complet** (éveil compris) : c'est la configuration dans laquelle ses performances ont été mesurées. L'éveil est retiré **après** la détection (colonne `toKeep`), avec le même masque `BooleanLiberal` que la CWT.

In [10]:

# ============================================================
# 1. CHEMINS : DONNEES SUR FORGETTING, COORDONNEES SUR AUDREY
# ============================================================

default_dpath = (
    r"\\10.69.168.1\crnldata\forgetting\Carla"
)

coordinates_path = Path(
    r"C:\Users\AudreyHay\Documents\Carla\Visual Code"
    r"\HayLabAnalysis\python\_LFP_coordinates_of_all_mice.csv"
)

# Charger les coordonnées des électrodes depuis Audrey
if coordinates_path.is_file():
    LFP_coordinates = pd.read_csv(coordinates_path)
    print("Coordonnées LFP chargées :", coordinates_path)
    print("Dimensions du CSV :", LFP_coordinates.shape)
else:
    raise FileNotFoundError(
        f"CSV des coordonnées introuvable : {coordinates_path}"
    )


# ============================================================
# 2. VERIFIER LE DOSSIER D'ENREGISTREMENT SUR FORGETTING
# ============================================================

# Réutiliser le chemin précédemment sélectionné s'il existe
try:
    %store -r dpath
except Exception:
    pass

if "dpath" not in globals() or not os.path.isdir(str(dpath)):
    dpath = default_dpath

if not os.path.isdir(dpath):
    raise FileNotFoundError(
        f"Dossier réseau inaccessible : {dpath}\n"
        "Vérifie le VPN et l'accès au lecteur réseau."
    )

folder_base = Path(dpath)
print("\nDossier de départ :", folder_base)


# ============================================================
# 3. RECHERCHER LES FICHIERS LFP
# ============================================================

# Formats reconnus par le script
file_names = [
    "DataFrame_rawdataDS.pkl",
    "RawDataChannelExtractedDS.npy",
    "continuous.dat",
]

# Priorité aux fichiers présents directement dans le dossier
LFPfile = None

for name in file_names:
    candidate = folder_base / name
    if candidate.is_file():
        LFPfile = candidate
        break

# Si aucun fichier n'est présent directement, chercher plus bas
if LFPfile is None:
    print("Recherche dans les sous-dossiers...")

    for root, dirs, files in os.walk(folder_base):
        # Eviter de parcourir des dossiers cachés ou temporaires
        dirs[:] = [
            d for d in dirs
            if not d.startswith(".")
        ]

        for name in file_names:
            if name in files:
                LFPfile = Path(root) / name
                break

        if LFPfile is not None:
            break

if LFPfile is None:
    raise FileNotFoundError(
        f"Aucun fichier LFP reconnu dans {folder_base} "
        "ni dans ses sous-dossiers.\n"
        "Fichiers recherchés : "
        + ", ".join(file_names)
    )

folder_base = LFPfile.parent

print("Fichier LFP trouvé :", LFPfile)
print("Dossier des données :", folder_base)


# ============================================================
# 4. CHARGER LES DONNEES SELON LE FORMAT
# ============================================================

if LFPfile.name == "DataFrame_rawdataDS.pkl":

    print("Chargement DataFrame_rawdataDS.pkl")

    LFPs_df = pd.read_pickle(LFPfile)
    DataRec = LFPs_df.to_numpy()

    samplerate = 1000
    rec_ch_list = np.array(LFPs_df.columns.values)

elif LFPfile.name == "RawDataChannelExtractedDS.npy":

    print("Chargement RawDataChannelExtractedDS.npy")

    DataRec = np.load(LFPfile, mmap_mode="r")

    samplerate = 1000
    rec_ch_list = np.arange(DataRec.shape[1])

    print("Pas de métadonnées de canaux disponibles dans ce fichier.")

elif LFPfile.name == "continuous.dat":

    print("Chargement continuous.dat")

    # Recherche du fichier de métadonnées Open Ephys
    possible_oebin = [
        folder_base / "structure.oebin",
        folder_base.parent / "structure.oebin",
        folder_base.parent.parent / "structure.oebin",
    ]

    oebin = next(
        (p for p in possible_oebin if p.is_file()),
        None
    )

    if oebin is None:
        raise FileNotFoundError(
            "structure.oebin introuvable. Chemins vérifiés :\n"
            + "\n".join(str(p) for p in possible_oebin)
        )

    with open(oebin, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    streams = metadata.get("continuous", [])

    if not streams:
        raise ValueError(
            f"Aucun flux continuous trouvé dans {oebin}"
        )

    # Trouver le flux correspondant au dossier choisi
    stream = next(
        (
            s for s in streams
            if s.get("folder_name", "").strip("/\\")
            == folder_base.name
        ),
        streams[0]
    )

    samplerate = stream["sample_rate"]
    numchannel = stream["num_channels"]

    # Numéros des canaux, indexés à partir de zéro
    channels = stream.get("channels", [])

    rec_ch_list = np.array([
        int("".join(
            c for c in ch["channel_name"] if c.isdigit()
        )) - 1
        for ch in channels
    ])

    # Lecture sans charger tout le fichier en mémoire
    raw = np.memmap(
        LFPfile,
        dtype=np.int16,
        mode="r"
    )

    n_complete = raw.size // numchannel
    remainder = raw.size % numchannel

    if remainder:
        print(
            f"Attention : {remainder} valeurs finales ignorées "
            "car elles ne forment pas un échantillon complet."
        )

    DataRec = raw[:n_complete * numchannel].reshape(
        -1, numchannel
    )

    print("Métadonnées Open Ephys chargées :", oebin)

else:
    raise ValueError(f"Format de fichier non pris en charge : {LFPfile}")


# ============================================================
# 5. INFORMATIONS SUR L'ENREGISTREMENT
# ============================================================

numchannel = DataRec.shape[1]

# 'fs' doit être défini auparavant dans ton notebook
if "fs" not in globals():
    raise NameError(
        "La variable fs n'est pas définie. "
        "Définis la fréquence d'échantillonnage cible avant cette cellule."
    )

n_total = int(DataRec.shape[0] * fs / samplerate)

print("\n========== RESUME LFP ==========")
print("Fichier :", LFPfile.name)
print("Fréquence d'origine :", samplerate, "Hz")
print("Fréquence cible fs :", fs, "Hz")
print("Nombre de canaux :", numchannel)
print("Canaux enregistrés :", list(rec_ch_list))
print("Durée approximative :", round(n_total / fs / 60), "minutes")
print("Dimensions des données :", DataRec.shape)
print("================================")

# Mémoriser le dossier sélectionné pour la prochaine exécution
dpath = str(folder_base)
%store dpath


Coordonnées LFP chargées : C:\Users\AudreyHay\Documents\Carla\Visual Code\HayLabAnalysis\python\_LFP_coordinates_of_all_mice.csv
Dimensions du CSV : (37, 1)

Dossier de départ : \\10.69.168.1\crnldata\forgetting\Carla\OpenEphysData\Sleep\10 mois\CarreRouge_CarreMarron_2026-09-22_16-07-50_Sleep_10mois
Recherche dans les sous-dossiers...
Fichier LFP trouvé : \\10.69.168.1\crnldata\forgetting\Carla\OpenEphysData\Sleep\10 mois\CarreRouge_CarreMarron_2026-09-22_16-07-50_Sleep_10mois\Record Node 101\experiment1\recording2\continuous\OE_FPGA_Acquisition_Board-100.Rhythm Data\continuous.dat
Dossier des données : \\10.69.168.1\crnldata\forgetting\Carla\OpenEphysData\Sleep\10 mois\CarreRouge_CarreMarron_2026-09-22_16-07-50_Sleep_10mois\Record Node 101\experiment1\recording2\continuous\OE_FPGA_Acquisition_Board-100.Rhythm Data
Chargement continuous.dat
Métadonnées Open Ephys chargées : \\10.69.168.1\crnldata\forgetting\Carla\OpenEphysData\Sleep\10 mois\CarreRouge_CarreMarron_2026-09-22_16-07-50_S

In [20]:
# --- Identification de la souris à partir du chemin ---
file_str = str(LFPfile)
found = sorted((file_str.find(name), name) for name in all_LFPcoordinates.index if name in file_str)
mouse_found = [name for _, name in found]      # dans l'ordre d'apparition dans le chemin

if len(mouse_found) == 0:
    raise ValueError(f"Aucun nom de souris du CSV ({list(all_LFPcoordinates.index)}) dans le chemin : {file_str}")

if len(mouse_found) > 1:
    if max(rec_ch_list) < 32:       # canaux < 32 : une seule souris enregistrée
        ID = 0
        mouse = mouse_found[id_in_path]
        print(f"/!/ Plusieurs noms dans le chemin {mouse_found} mais une seule souris enregistrée. Le n°{id_in_path + 1} est choisi : {mouse}")
    else:                           # plusieurs souris enregistrées en même temps
        ID_choice = 1
        mouse = mouse_found[ID_choice]
        print(f"/!/ Plusieurs souris enregistrées {mouse_found}. Le n°{ID + 1} est choisi : {mouse}")
elif max(rec_ch_list) >= 32:
    ID = ID_choice
    mouse = mouse_found[0]
    print(f"/!/ Plusieurs souris enregistrées mais un seul nom dans le chemin. Souris : {mouse}, entrée OE n°{ID + 1}")
else:
    ID = int(rec_ch_list[0] / 32)
    mouse = mouse_found[0]
    print(f"Une seule souris enregistrée = {mouse}, entrée OE n°{ID + 1}")

mice = mouse    # nom utilisé par la détection et les fichiers de sortie
for region in ('CA1', 'EMG'):
    assert region in all_LFPcoordinates.columns, f"Colonne '{region}' absente du CSV"

/!/ Plusieurs souris enregistrées ['CarreRouge', 'CarreMarron']. Le n°1 est choisi : CarreMarron


In [21]:
def get_region_signal(region, n_rows):
    """Signal d'une région : canal1 - canal2 si 2 canaux dans le CSV (ex '5_6'), sinon canal seul. Retourné à fs."""
    coord = all_LFPcoordinates.loc[mouse, region]
    if coord in ('nan', ''):
        raise ValueError(f"Pas de canal '{region}' pour {mouse} dans le CSV")
    # dans le CSV les numéros vont de 0 à 31 pour l'entrée OE de la souris -> numéro absolu = + ID*32
    chans = [int(float(c)) + ID * 32 for c in coord.split('_')[:2]]
    cols = []
    for ch in chans:
        where = np.where(rec_ch_list == ch)[0]
        if len(where) == 0:
            raise ValueError(f"{region} : canal {ch} non enregistré. Canaux enregistrés : {list(rec_ch_list)}")
        cols.append(int(where[0]))
    sig = np.asarray(DataRec[:n_rows, cols[0]], dtype=np.float32)
    if len(cols) > 1:
        sig = sig - np.asarray(DataRec[:n_rows, cols[1]], dtype=np.float32)
    if samplerate != fs:
        sig = signal.resample_poly(sig, int(fs), int(samplerate)).astype(np.float32)
    return sig, chans

n_wanted = n_total if not test_minutes else min(n_total, int(test_minutes * 60 * fs))
n_rows = int(round(n_wanted * samplerate / fs))

CA1, CA1_chans = get_region_signal('CA1', n_rows)
EMG, EMG_chans = get_region_signal('EMG', n_rows)
n = min(len(CA1), len(EMG), n_wanted)
CA1, EMG = CA1[:n], EMG[:n]

emg_boolean_file = os.path.join(dpath, 'EMGframeBoolean.pkl')
EMGboolean = pd.read_pickle(emg_boolean_file) if os.path.isfile(emg_boolean_file) else None
print(f"{mouse} | CA1 = ch{CA1_chans[0]} - ch{CA1_chans[-1]}, EMG = ch{EMG_chans}, {n / fs / 60:.1f} min",
      "| masque d'éveil :", "trouvé" if EMGboolean is not None else "absent (toKeep = True partout)")

CarreMarron | CA1 = ch14 - ch15, EMG = ch[10], 10.0 min | masque d'éveil : absent (toKeep = True partout)


### Détection

- `host = 'local'` : le calcul se fait dans cette cellule.
- `host = 'CRNLcluster'` : cette cellule envoie le calcul au cluster et rend la main tout de suite ; on peut fermer le notebook. La cellule suivante récupère le résultat : la relancer jusqu'à ce qu'elle ne dise plus que le calcul est en cours (après un redémarrage du kernel, relancer d'abord les cellules jusqu'à *Chargement*).

In [22]:
session_id = os.path.basename(os.path.normpath(dpath))
if host == 'local':
    result = detect_swr(CA1, EMG, sampling_rate_hz=fs, output_dir=dpath, output_suffix=suffix,
                        mouse_id=mice, session_id=session_id)
else:
    swr_job = detect_swr(CA1, EMG, sampling_rate_hz=fs, output_dir=dpath, output_suffix=suffix,
                         mouse_id=mice, session_id=session_id, host=host, remote_activate_cmd=cluster_activate)
    %store swr_job
    print("Calcul envoyé au cluster : lancer la cellule suivante pour récupérer le résultat.")

[run_ml_detection_on_worker] [CarreMarron/OE_FPGA_Acquisition_Board-100.Rhythm Data] starting: 600,000 samples, checkpoint=cnn_bilstm_best.pt, device=cpu
[haylab_oscillation_models] [CarreMarron/OE_FPGA_Acquisition_Board-100.Rhythm Data] starting SWR detection: 600,000 samples (600.0s) @ 1000 Hz, checkpoint=cnn_bilstm_best.pt
[haylab_oscillation_models] [CarreMarron/OE_FPGA_Acquisition_Board-100.Rhythm Data] loading checkpoint on cpu...
[haylab_oscillation_models] [CarreMarron/OE_FPGA_Acquisition_Board-100.Rhythm Data] running inference: 1,200 windows, batch_size=64
[haylab_oscillation_models] inference progress: 896/2,399 active windows (37.3%) | elapsed 5.4s | ETA 9.0s | 167.1 windows/s | 0 zero windows skipped
[haylab_oscillation_models] inference progress: 1,920/2,399 active windows (80.0%) | elapsed 10.4s | ETA 2.6s | 185.3 windows/s | 0 zero windows skipped
[haylab_oscillation_models] inference progress: 2,399/2,399 active windows (100.0%) | elapsed 13.2s | ETA 0.0s | 181.6 windo

In [ ]:
if host != 'local':
    %store -r swr_job
    result = collect_swr(swr_job)   # TimeoutError = pas encore fini : relancer cette cellule plus tard
ML_SWR = result['dataframe'].copy()
meta = result['metadata']
All = DataRec
print(f"{len(ML_SWR)} SWR en {meta['inference_seconds']:.0f} s",
      f"(≈ {meta['inference_seconds'] / (n / fs) * All.shape[0] / fs / 60:.1f} min estimées pour la session entière)")
print({k: meta[k] for k in ('haylab_tools_version', 'oscillation_models_version', 'checkpoint_md5')})

### Retrait de l'éveil et sauvegarde au format de la CWT

Fichiers écrits dans le dossier de la session (avec `_test10min` en plus dans le nom en mode test) :
- `SWRproperties_ml{ID}.csv` : même format que le CSV CWT, avec `toKeep = False` pour les SWR pendant l'éveil ;
- `SWRs_<souris>_ml{ID}.csv/.pkl/_metadata.json/_trace.npz` : sortie brute du modèle. Le `_metadata.json` garde la version du modèle utilisée, ne pas le supprimer.
- avec `host = 'CRNLcluster'`, en plus : `SWRs_<souris>_ml{ID}_signals.npz`, les signaux envoyés au cluster (supprimable une fois le résultat récupéré).

In [23]:
if EMGboolean is not None:
    wake = np.asarray(EMGboolean['BooleanLiberal'])[:n][ML_SWR['peak time'].to_numpy(dtype=int)]
    ML_SWR.insert(0, 'inWakeLiberal', wake)
    ML_SWR.insert(0, 'toKeep', ~wake)
else:
    ML_SWR.insert(0, 'toKeep', True)

filenameOutput = os.path.join(dpath, f'SWRproperties{suffix}.csv')
ML_SWR.to_csv(filenameOutput, sep=',')
print(f"{ML_SWR['toKeep'].sum()} SWR gardées sur {len(ML_SWR)} -> {filenameOutput}")

13 SWR gardées sur 13 -> \\10.69.168.1\crnldata\forgetting\Carla\OpenEphysData\Sleep\10 mois\CarreRouge_CarreMarron_2026-09-22_16-07-50_Sleep_10mois\Record Node 101\experiment1\recording2\continuous\OE_FPGA_Acquisition_Board-100.Rhythm Data\SWRproperties_mlCarreRouge_test10min.csv


### Comparaison avec la détection CWT (si elle existe)

Une SWR est « commune » si les pics des deux méthodes sont à moins de `tol_ms` l'un de l'autre. Seules les SWR `toKeep` sont comparées : si le CSV CWT a été trié à la main, c'est ce tri qui sert de référence. À regarder en priorité : les SWR détectées autour des **stimulations opto**, car le modèle n'a jamais vu ces artefacts à l'entraînement.

In [24]:
tol_ms = 50
cwt_file = os.path.join(dpath, f'SWRproperties{ID}.csv')

def has_match(a, b, tol):
    if len(b) == 0:
        return np.zeros(len(a), dtype=bool)
    b = np.sort(b)
    idx = np.searchsorted(b, a)
    left = np.abs(a - b[np.clip(idx - 1, 0, len(b) - 1)])
    right = np.abs(b[np.clip(idx, 0, len(b) - 1)] - a)
    return np.minimum(left, right) <= tol

if os.path.isfile(cwt_file):
    CWT = pd.read_csv(cwt_file, index_col=0)
    cwt_peaks = CWT.loc[CWT['toKeep'], 'peak time'].to_numpy()
    cwt_peaks = cwt_peaks[cwt_peaks < n]
    ml_peaks = ML_SWR.loc[ML_SWR['toKeep'], 'peak time'].to_numpy(dtype=int)
    found = has_match(cwt_peaks, ml_peaks, tol_ms)
    confirmed = has_match(ml_peaks, cwt_peaks, tol_ms)
    print(f"CWT gardées : {len(cwt_peaks)} | ML gardées : {len(ml_peaks)}")
    print(f"SWR CWT retrouvées par le ML : {found.sum()} ({100 * found.mean() if len(found) else 0:.0f} %)")
    print(f"SWR ML absentes de la CWT : {(~confirmed).sum()}")
else:
    print(f"Pas de détection CWT trouvée ({os.path.basename(cwt_file)})")

Pas de détection CWT trouvée (SWRproperties0.csv)


### Visualisation

Traces : CA1 brut, CA1 filtré 120-200 Hz, probabilité donnée par le modèle, EMG. Points blancs : pics ; noirs : début et fin.

In [ ]:
%%capture
%gui qt
from ephyviewer import mkQApp, MainViewer, TraceViewer, EventList, InMemoryEventSource, AnalogSignalSourceWithScatter

app = mkQApp()

all_events = []
for name, sel in [('All', np.ones(len(ML_SWR), dtype=bool)), ('toKeep', ML_SWR['toKeep'].to_numpy(dtype=bool))]:
    all_events.append({'time': ML_SWR.loc[sel, 'peak time'].values / fs,
                       'label': [f'SWR {i}' for i in ML_SWR.index[sel]], 'name': name})
source_ev = InMemoryEventSource(all_events=all_events)

sos = signal.butter(4, [120, 200], btype='band', fs=fs, output='sos')
filt_CA1 = signal.sosfiltfilt(sos, CA1)
proba = np.load(result['output_trace_npz'])['predictions_raw']

def robust_z(x):
    sub = x[::max(1, len(x) // 1_000_000)]
    med = np.median(sub)
    mad = np.median(np.abs(sub - med)) * 1.4826
    return ((x - med) / (mad if mad > 0 else 1)).astype(np.float32)

# Layout fixed here, viewer kept in real_scale: ephyviewer's by_channel autoscale divides by each
# trace's MAD, which is ~0 for the mostly-zero ML probability and blows it up across all traces.
traces = [('CA1', robust_z(CA1) / 10, '#88FF88'),
          ('FiltCA1', robust_z(filt_CA1) / 10, '#0055ff'),
          ('ML proba', proba - 0.5, '#ff5500'),
          ('EMG', robust_z(EMG) / 10, '#888888')]
combined = np.stack([sig + 1.5 * (len(traces) - 1 - k) for k, (_, sig, _) in enumerate(traces)], axis=1)

scatter_indexes = {0: ML_SWR['peak time'].to_numpy(dtype=int), 1: ML_SWR['start time'].to_numpy(dtype=int), 2: ML_SWR['end time'].to_numpy(dtype=int)}
scatter_channels = {0: [1, 2], 1: [0, 1], 2: [0, 1]}
source = AnalogSignalSourceWithScatter(combined, float(fs), 0., scatter_indexes, scatter_channels,
                                       scatter_colors={0: '#FFFFFF', 1: '#FFFFFF', 2: '#222222'},
                                       channel_names=[name for name, _, _ in traces])

win = MainViewer(debug=True, show_auto_scale=True)
view1 = TraceViewer(source=source)
view1.params['scale_mode'] = 'real_scale'
view1.params['display_labels'] = True
for k, (_, _, color) in enumerate(traces):
    view1.by_channel_params[f'ch{k}', 'color'] = color
view1.auto_scale()
view2 = EventList(source=source_ev, name='event')
win.add_view(view1)
win.add_view(view2, location='bottom', orientation='horizontal')
win.show()

refresh duration for  0.0 s
refresh duration for  0.0 s
refresh duration for event 0.0 s
refresh duration for  0.0010023117065429688 s
refresh duration for event 0.0 s
refresh duration for  0.0 s
refresh duration for event 0.0 s
refresh duration for  0.0 s
refresh duration for  0.0 s
refresh duration for event 0.0 s
refresh duration for  0.0009520053863525391 s
refresh duration for event 0.0 s
refresh duration for  0.002008676528930664 s
refresh duration for event 0.0 s
refresh duration for  0.0 s
refresh duration for event 0.0 s
refresh duration for  0.0 s
refresh duration for  0.0 s
refresh duration for event 0.0 s
refresh duration for  0.0 s
refresh duration for event 0.0 s
refresh duration for  0.0 s
refresh duration for event 0.0 s
refresh duration for  0.0 s
refresh duration for  0.0 s
refresh duration for event 0.0 s
save_all_settings
